# TrendPulse – What's Actually Trending Right Now

## Part 1 – Python Foundations & Data Structures

### Project Objective
To analyze trending YouTube videos from multiple countries using Python fundamentals, data structures, functions, loops and rule-based classification.

### Dataset
Trending YouTube Video Statistics Dataset.

## 1. Dataset Information

This project uses daily trending YouTube video statistics from multiple countries.

The dataset contains information about:
- Video ID
- Video title
- Channel name
- Category ID
- Views
- Likes
- Dislikes
- Comment count
- Trending date
- Publish time

## 2. Configuration and Setup

In this section, I will configure the dataset file paths, country names and classification thresholds required for the TrendPulse project.

I will also display the execution date and time using Python f-strings.

In [43]:
import pandas as pd
import numpy as np
import os
from datetime import datetime

from google.colab import drive
drive.mount('/content/drive')

DATA_PATH = "/content/drive/MyDrive/trendpulse/trendpulse/part1_python_foundations/data"

COUNTRIES = ["US", "GB", "IN"]

VIRAL_THRESHOLD = 1_000_000
STEADY_THRESHOLD = 100_000

RUN_TIME = datetime.now()

print("=" * 50)
print("       TRENDPULSE - PART 1")
print("=" * 50)
print(f"Countries: {COUNTRIES}")
print(f"Run started: {RUN_TIME.strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 50)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
       TRENDPULSE - PART 1
Countries: ['US', 'GB', 'IN']
Run started: 2026-11-08 09:43:32


In [44]:
def load_country(path, country, max_rows=None):

    df = pd.read_csv(path, nrows=max_rows)

    required_columns = [
        "video_id",
        "trending_date",
        "title",
        "channel_title",
        "category_id",
        "publish_time",
        "views",
        "likes",
        "dislikes",
        "comment_count",
        "comments_disabled",
        "ratings_disabled",
        "video_error_or_removed"
    ]

    missing_columns = [
        column for column in required_columns
        if column not in df.columns
    ]

    if missing_columns:
        raise ValueError(f"Missing columns: {missing_columns}")

    df["country"] = country

    records = df.to_dict("records")

    print(f"{country}: {len(records)} records loaded")

    return records

In [45]:
all_records = []

for country in COUNTRIES:

    file_path = os.path.join(
        DATA_PATH,
        f"{country}videos.csv"
    )

    records = load_country(file_path, country)

    all_records.extend(records)

print("Total records:", len(all_records))

US: 16080 records loaded
GB: 15075 records loaded
IN: 15075 records loaded
Total records: 46230


In [46]:
categories_df = pd.read_csv(
    os.path.join(DATA_PATH, "categories.csv")
)

category_names = {}

for row in categories_df.to_dict("records"):

    category_names[int(row["category_id"])] = row["category_name"]

print(category_names)

{1: 'Film & Animation', 2: 'Autos & Vehicles', 10: 'Music', 15: 'Pets & Animals', 17: 'Sports', 19: 'Travel & Events', 20: 'Gaming', 22: 'People & Blogs', 23: 'Comedy', 24: 'Entertainment', 25: 'News & Politics', 26: 'Howto & Style', 27: 'Education', 28: 'Science & Technology', 29: 'Nonprofits & Activism', 30: 'Movies', 43: 'Shows'}


In [47]:
parse_failures = {
    "views": 0,
    "likes": 0,
    "dislikes": 0,
    "comment_count": 0,
    "comments_disabled": 0,
    "ratings_disabled": 0,
    "video_error_or_removed": 0,
    "trending_date": 0,
    "publish_time": 0
}

In [49]:
def convert_integer(value, field):

    try:

        if pd.isna(value) or str(value).strip() == "":
            raise ValueError("Missing value")

        return int(float(value))

    except (ValueError, TypeError):

        parse_failures[field] += 1

        return 0

In [51]:
def convert_boolean(value, field):

    try:

        if isinstance(value, bool):
            return value

        value = str(value).strip().lower()

        if value in ["true", "1"]:
            return True

        elif value in ["false", "0"]:
            return False

        else:
            raise ValueError("Invalid boolean")

    except (ValueError, TypeError):

        parse_failures[field] += 1

        return False

In [52]:
def convert_trending_date(value):

    try:

        return datetime.strptime(
            str(value).strip(),
            "%y.%d.%m"
        )

    except (ValueError, TypeError):

        parse_failures["trending_date"] += 1

        return None

In [53]:
def convert_publish_time(value):

    try:

        return datetime.fromisoformat(
            str(value).strip().replace("Z", "+00:00")
        )

    except (ValueError, TypeError):

        parse_failures["publish_time"] += 1

        return None

In [54]:
cleaned_records = []

for record in all_records:

    record["views"] = convert_integer(
        record["views"], "views"
    )

    record["likes"] = convert_integer(
        record["likes"], "likes"
    )

    record["dislikes"] = convert_integer(
        record["dislikes"], "dislikes"
    )

    record["comment_count"] = convert_integer(
        record["comment_count"], "comment_count"
    )

    record["comments_disabled"] = convert_boolean(
        record["comments_disabled"], "comments_disabled"
    )

    record["ratings_disabled"] = convert_boolean(
        record["ratings_disabled"], "ratings_disabled"
    )

    record["video_error_or_removed"] = convert_boolean(
        record["video_error_or_removed"], "video_error_or_removed"
    )

    record["trending_date"] = convert_trending_date(
        record["trending_date"]
    )

    record["publish_time"] = convert_publish_time(
        record["publish_time"]
    )

    cleaned_records.append(record)

In [55]:
print("PARSING FAILURE REPORT")

for field, count in parse_failures.items():

    print(f"{field}: {count} failures")

PARSING FAILURE REPORT
views: 0 failures
likes: 12 failures
dislikes: 0 failures
comment_count: 36 failures
comments_disabled: 0 failures
ratings_disabled: 0 failures
video_error_or_removed: 0 failures
trending_date: 0 failures
publish_time: 0 failures


In [56]:
for record in cleaned_records:

    category_id = record["category_id"]

    record["category_name"] = category_names.get(
        int(category_id),
        "Unknown"
    )

In [57]:
video_records = cleaned_records

print(type(video_records))
print(len(video_records))
print(video_records[0])

<class 'list'>
46230
{'video_id': 'g_N-w8LeAi8', 'trending_date': datetime.datetime(2017, 11, 14, 0, 0), 'title': 'Honest Football Prank LEAKED', 'channel_title': 'QuietSpeech10', 'category_id': 22, 'publish_time': datetime.datetime(2017, 11, 14, 7, 4, 28, tzinfo=datetime.timezone.utc), 'tags': 'Monsoon|Robots', 'views': 48555, 'likes': 2189, 'dislikes': 180, 'comment_count': 136, 'thumbnail_link': 'https://i.ytimg.com/vi/g_N-w8LeAi8/default.jpg', 'comments_disabled': False, 'ratings_disabled': False, 'video_error_or_removed': False, 'description': 'Watch this video. Subscribe for more content about chess. Business enquiries in the channel about page.', 'country': 'US', 'category_name': 'People & Blogs'}


In [80]:
for field in ["views", "likes", "dislikes", "comment_count"]:
    missing_count = 0

    for video in video_records:
        if video[field] is None:
            missing_count += 1

    print(field, "Missing values:", missing_count)

views Missing values: 0
likes Missing values: 0
dislikes Missing values: 0
comment_count Missing values: 0


In [81]:
boolean_fields = [
    "comments_disabled",
    "ratings_disabled",
    "video_error_or_removed"
]

for field in boolean_fields:
    true_count = 0
    false_count = 0

    for video in video_records:
        if video[field] == True:
            true_count += 1
        else:
            false_count += 1

    print(field)
    print("True:", true_count)
    print("False:", false_count)
    print()

comments_disabled
True: 1048
False: 45182

ratings_disabled
True: 544
False: 45686

video_error_or_removed
True: 92
False: 46138



In [82]:
trending_date_failures = 0
publish_time_failures = 0

for video in video_records:

    if video["trending_date"] is None:
        trending_date_failures += 1

    if video["publish_time"] is None:
        publish_time_failures += 1

print("Trending date failures:", trending_date_failures)
print("Publish time failures:", publish_time_failures)

Trending date failures: 0
Publish time failures: 0


In [83]:
seen_videos = set()
duplicate_count = 0

for video in video_records:

    video_key = (
        video["country"],
        video["video_id"],
        video["trending_date"]
    )

    if video_key in seen_videos:
        duplicate_count += 1
    else:
        seen_videos.add(video_key)

print("Duplicate records:", duplicate_count)
print("Unique records:", len(seen_videos))

Duplicate records: 230
Unique records: 46000


In [84]:
unique_records = []
seen_videos = set()

for video in video_records:
    key = (video["country"], video["video_id"], video["trending_date"])

    if key not in seen_videos:
        unique_records.append(video)
        seen_videos.add(key)

print("Records before:", len(video_records))
print("Records after:", len(unique_records))

Records before: 46230
Records after: 46000


In [85]:
country_counts = {}

for video in unique_records:
    country = video["country"]

    if country not in country_counts:
        country_counts[country] = 0

    country_counts[country] += 1

print(country_counts)

{'US': 16000, 'GB': 15000, 'IN': 15000}


In [86]:
country_views = {}

for video in unique_records:
    country = video["country"]

    if country not in country_views:
        country_views[country] = 0

    country_views[country] += video["views"]

print(country_views)

{'US': 16222470043, 'GB': 23760619750, 'IN': 14281511461}


In [87]:
channel_counts = {}

for video in unique_records:
    channel = video["channel_title"]

    if channel not in channel_counts:
        channel_counts[channel] = 0

    channel_counts[channel] += 1

top_channels = sorted(
    channel_counts.items(),
    key=lambda item: item[1],
    reverse=True
)

print(top_channels[:10])

[('QuietChallenge75', 2451), ('OfficialReview41', 1577), ('UltimateUnboxing65', 1428), ('InsaneReaction46', 1226), ('Brand NewChallenge45', 1172), ('CrazyTrailer91', 1012), ('FinalTrailer17', 904), ('UltimateSpeech20', 766), ('WildHighlights7', 682), ('Brand NewSecret51', 681)]


In [88]:
country_category = {}

for video in unique_records:
    country = video["country"]
    category = video["category_name"]

    if country not in country_category:
        country_category[country] = {}

    if category not in country_category[country]:
        country_category[country][category] = 0

    country_category[country][category] += 1

for country, categories in country_category.items():
    print("\n", country)
    print(categories)


 US
{'People & Blogs': 1419, 'News & Politics': 751, 'Music': 3039, 'Education': 559, 'Entertainment': 4351, 'Howto & Style': 709, 'Comedy': 1359, 'Gaming': 799, 'Travel & Events': 262, 'Sports': 662, 'Film & Animation': 819, 'Autos & Vehicles': 403, 'Pets & Animals': 318, 'Science & Technology': 550}

 GB
{'Science & Technology': 469, 'Howto & Style': 586, 'Comedy': 937, 'Entertainment': 2826, 'Music': 4990, 'Gaming': 603, 'People & Blogs': 1292, 'Autos & Vehicles': 345, 'Education': 467, 'Sports': 676, 'News & Politics': 533, 'Film & Animation': 710, 'Pets & Animals': 309, 'Travel & Events': 257}

 IN
{'Music': 2281, 'Film & Animation': 768, 'Entertainment': 5093, 'Comedy': 962, 'People & Blogs': 1093, 'Education': 619, 'News & Politics': 1138, 'Travel & Events': 190, 'Howto & Style': 530, 'Sports': 583, 'Pets & Animals': 286, 'Autos & Vehicles': 259, 'Gaming': 655, 'Science & Technology': 543}


In [89]:
country_name = "Global Country"

def show_country():
    local_name = "Local Country"

    print("Inside function:", local_name)
    print("Global variable:", country_name)

show_country()

print("Outside function:", country_name)

Inside function: Local Country
Global variable: Global Country
Outside function: Global Country


In [90]:
def title_features(title):

    uppercase_words = 0
    exclamation_marks = 0
    question_marks = 0

    words = title.split()

    for word in words:
        if word.isupper():
            uppercase_words += 1

    for char in title:
        if char == "!":
            exclamation_marks += 1
        elif char == "?":
            question_marks += 1

    word_count = 0

    for i in range(len(words)):
        word_count += 1

    return {
        "uppercase_words": uppercase_words,
        "exclamation_marks": exclamation_marks,
        "question_marks": question_marks,
        "word_count": word_count
    }


print(title_features("WOW! THIS IS AMAZING?"))

{'uppercase_words': 4, 'exclamation_marks': 1, 'question_marks': 1, 'word_count': 4}


In [91]:
def classify_video(video):

    views = video["views"]
    likes = video["likes"]
    dislikes = video["dislikes"]

    if video["ratings_disabled"]:
        return "Unrated"

    if views == 0 or likes + dislikes == 0:
        return "Unrated"

    like_ratio = likes / (likes + dislikes)

    if views >= 1_000_000 and like_ratio >= 0.80:
        return "Viral Hit"

    elif views >= 100_000 and like_ratio >= 0.65 and not video["comments_disabled"]:
        return "Steady Performer"

    elif views >= 100_000 and like_ratio < 0.50:
        return "Divisive"

    else:
        return "Low Engagement"

In [92]:
classification_counts = {}

for video in unique_records:
    label = classify_video(video)

    if label not in classification_counts:
        classification_counts[label] = 0

    classification_counts[label] += 1

print(classification_counts)

{'Low Engagement': 9619, 'Steady Performer': 26364, 'Viral Hit': 9444, 'Divisive': 34, 'Unrated': 539}


In [93]:
def count_values(records, field):
    counts = {}

    for record in records:
        value = record[field]

        if value not in counts:
            counts[value] = 0

        counts[value] += 1

    return counts


# Count videos by country
print(count_values(unique_records, "country"))

# Count videos by category
print(count_values(unique_records, "category_name"))

{'US': 16000, 'GB': 15000, 'IN': 15000}
{'People & Blogs': 3804, 'News & Politics': 2422, 'Music': 10310, 'Education': 1645, 'Entertainment': 12270, 'Howto & Style': 1825, 'Comedy': 3258, 'Gaming': 2057, 'Travel & Events': 709, 'Sports': 1921, 'Film & Animation': 2297, 'Autos & Vehicles': 1007, 'Pets & Animals': 913, 'Science & Technology': 1562}


In [58]:
video_tuples = []

for video in video_records:

    video_info = (
        video["video_id"],
        video["trending_date"],
        video["views"],
        video["likes"]
    )

    video_tuples.append(video_info)

print(video_tuples[:5])

[('g_N-w8LeAi8', datetime.datetime(2017, 11, 14, 0, 0), 48555, 2189), ('Ugyql2NlPAd', datetime.datetime(2017, 11, 14, 0, 0), 198385, 1159), ('8fE19jkxJ9n', datetime.datetime(2017, 11, 14, 0, 0), 231656, 8742), ('WOY0nn7HCsO', datetime.datetime(2017, 11, 14, 0, 0), 8083, 176), ('2Lc_Cs5VVr_', datetime.datetime(2017, 11, 14, 0, 0), 373161, 11305)]


In [59]:
country_video_ids = {}

for country in COUNTRIES:

    country_video_ids[country] = set()

    for video in video_records:

        if video["country"] == country:

            country_video_ids[country].add(
                video["video_id"]
            )

for country, ids in country_video_ids.items():

    print(country, "Unique videos:", len(ids))

US Unique videos: 5068
GB Unique videos: 4113
IN Unique videos: 5878


In [60]:
us_ids = country_video_ids["US"]
gb_ids = country_video_ids["GB"]
in_ids = country_video_ids["IN"]

us_gb = us_ids.intersection(gb_ids)
us_in = us_ids.intersection(in_ids)
gb_in = gb_ids.intersection(in_ids)

multiple_country_videos = us_gb.union(us_in, gb_in)

print("Videos trending in multiple countries:",
      len(multiple_country_videos))

Videos trending in multiple countries: 340


In [61]:
us_only = us_ids - (gb_ids | in_ids)

gb_only = gb_ids - (us_ids | in_ids)

in_only = in_ids - (us_ids | gb_ids)

print("US-only videos:", len(us_only))
print("GB-only videos:", len(gb_only))
print("IN-only videos:", len(in_only))

US-only videos: 4834
GB-only videos: 3873
IN-only videos: 5652


In [62]:
channel_counts = {}

for video in video_records:

    channel = video["channel_title"]

    if channel not in channel_counts:

        channel_counts[channel] = 0

    channel_counts[channel] += 1

In [63]:
sorted_channels = sorted(
    channel_counts.items(),
    key=lambda item: item[1],
    reverse=True
)

top_15_channels = sorted_channels[:15]

for channel, count in top_15_channels:

    print(channel, ":", count)

QuietChallenge75 : 2464
OfficialReview41 : 1586
UltimateUnboxing65 : 1433
InsaneReaction46 : 1233
Brand NewChallenge45 : 1175
CrazyTrailer91 : 1018
FinalTrailer17 : 911
UltimateSpeech20 : 770
Brand NewSecret51 : 684
WildHighlights7 : 682
SurprisingHighlights41 : 650
FinalBreakdown11 : 643
CrazyRecipe67 : 603
QuietInterview9 : 570
UltimateVlog96 : 565


In [64]:
country_category_summary = {}

for video in video_records:

    country = video["country"]
    category = video["category_name"]

    if country not in country_category_summary:

        country_category_summary[country] = {}

    if category not in country_category_summary[country]:

        country_category_summary[country][category] = 0

    country_category_summary[country][category] += 1

In [65]:
for country, categories in country_category_summary.items():

    print(f"\nCountry: {country}")

    for category, count in list(categories.items())[:5]:

        print(f"{category}: {count}")


Country: US
People & Blogs: 1423
News & Politics: 755
Music: 3052
Education: 561
Entertainment: 4371

Country: GB
Science & Technology: 470
Howto & Style: 588
Comedy: 941
Entertainment: 2844
Music: 5013

Country: IN
Music: 2298
Film & Animation: 770
Entertainment: 5117
Comedy: 966
People & Blogs: 1098


In [66]:
global_message = "Welcome to TrendPulse"

def demonstrate_scope():

    local_message = "Python Foundations"

    print("Inside function:")
    print(global_message)
    print(local_message)

demonstrate_scope()

Inside function:
Welcome to TrendPulse
Python Foundations


In [67]:
try:

    print(local_message)

except NameError:

    print("local_message cannot be accessed outside the function")

local_message cannot be accessed outside the function


In [79]:
def title_features(title):

    uppercase_words = 0
    exclamation_marks = 0
    question_marks = 0

    words = title.split()

    # Count uppercase words
    for word in words:
        if word.isupper():
            uppercase_words += 1

    # Count punctuation marks
    for character in title:
        if character == "!":
            exclamation_marks += 1
        elif character == "?":
            question_marks += 1

    # Count words using range and index
    word_count = 0

    for i in range(len(words)):
        current_word = words[i]
        word_count += 1

    return {
        "uppercase_words": uppercase_words,
        "exclamation_marks": exclamation_marks,
        "question_marks": question_marks,
        "word_count": word_count
    }

In [69]:
example_title = "WOW! THIS IS AMAZING?"

print(title_features(example_title))

{'uppercase_words': 4, 'exclamation_marks': 1, 'question_marks': 1, 'word_count': 4}


In [70]:
index = 0

while index < len(video_records):

    video = video_records[index]

    index += 1

    if not video["title"]:

        continue

    if video["views"] > VIRAL_THRESHOLD:

        print("First video above threshold:")
        print(video["title"])
        print(video["views"])

        break

First video above threshold:
Official Cats Speech EXCLUSIVE
2841278


In [71]:
def classify_video(record):

    views = record["views"]
    likes = record["likes"]
    dislikes = record["dislikes"]

    comments_disabled = record["comments_disabled"]
    ratings_disabled = record["ratings_disabled"]

    total_reactions = likes + dislikes

    # Handle unrated videos
    if ratings_disabled or (
        views == 0 and total_reactions == 0
    ):

        return "Unrated"

    # Prevent division by zero
    if total_reactions == 0:

        return "Unrated"

    like_ratio = likes / total_reactions

    # Nested conditions
    if views >= VIRAL_THRESHOLD:

        if like_ratio >= 0.80:

            return "Viral Hit"

    elif views >= STEADY_THRESHOLD:

        if like_ratio < 0.50:

            return "Divisive"

        elif like_ratio >= 0.65 and not comments_disabled:

            return "Steady Performer"

    return "Low Engagement"

In [72]:
label_counts = {}

for video in video_records:

    label = classify_video(video)

    video["classification"] = label

    if label not in label_counts:

        label_counts[label] = 0

    label_counts[label] += 1

In [73]:
print("VIDEO CLASSIFICATION SUMMARY")

for label, count in label_counts.items():

    print(f"{label}: {count}")

VIDEO CLASSIFICATION SUMMARY
Low Engagement: 9738
Steady Performer: 26420
Viral Hit: 9497
Divisive: 31
Unrated: 544


In [74]:
example_video = {
    "views": 2000000,
    "likes": 90000,
    "dislikes": 10000,
    "comments_disabled": False,
    "ratings_disabled": False
}

In [75]:
print(classify_video(example_video))

Viral Hit


In [76]:
def count_values(records, key):

    counts = {}

    for record in records:

        value = record[key]

        if value not in counts:

            counts[value] = 0

        counts[value] += 1

    return counts

In [77]:
country_counts = count_values(video_records, "country")

category_counts = count_values(video_records, "category_name")

classification_counts = count_values(video_records, "classification")

print("Country counts:", country_counts)

print("\nCategory counts:", category_counts)

print("\nClassification counts:", classification_counts)

Country counts: {'US': 16080, 'GB': 15075, 'IN': 15075}

Category counts: {'People & Blogs': 3820, 'News & Politics': 2432, 'Music': 10363, 'Education': 1652, 'Entertainment': 12332, 'Howto & Style': 1837, 'Comedy': 3273, 'Gaming': 2073, 'Travel & Events': 711, 'Sports': 1930, 'Film & Animation': 2308, 'Autos & Vehicles': 1009, 'Pets & Animals': 918, 'Science & Technology': 1572}

Classification counts: {'Low Engagement': 9738, 'Steady Performer': 26420, 'Viral Hit': 9497, 'Divisive': 31, 'Unrated': 544}


In [78]:
output_path = os.path.join(
    DATA_PATH,
    "cleaned_video_records.csv"
)

cleaned_df = pd.DataFrame(video_records)

cleaned_df.to_csv(output_path, index=False)

print("Cleaned dataset saved successfully!")

Cleaned dataset saved successfully!


In [94]:
!git --version


git version 2.43.0


In [96]:
!git config --global user.name "Karthika Rajendiran"
!git config --global user.email "karthikarajendiran2004@gmail.com"

In [97]:
!git clone https://github.com/karthikarajendiran2004-cyber/TrendPulse.git

Cloning into 'TrendPulse'...
remote: Enumerating objects: 3, done.
remote: Counting objects: 100% (3/3), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 3 (delta 0), reused 3 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (3/3), done.


In [98]:
import shutil

source = "/content/drive/MyDrive/trendpulse/trendpulse/part1_python_foundations"
destination = "/content/TrendPulse/part1_python_foundations"

shutil.copytree(source, destination, dirs_exist_ok=True)

print("Part 1 copied successfully!")

Part 1 copied successfully!


In [99]:
%cd /content/TrendPulse
!git status

/content/TrendPulse
On branch main
Your branch is up to date with 'origin/main'.

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	part1_python_foundations/

nothing added to commit but untracked files present (use "git add" to track)


In [100]:
%cd /content/TrendPulse
!git add part1_python_foundations/
!git status

/content/TrendPulse
On branch main
Your branch is up to date with 'origin/main'.

Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	new file:   part1_python_foundations/README.md
	new file:   part1_python_foundations/data/GBvideos.csv
	new file:   part1_python_foundations/data/INvideos.csv
	new file:   part1_python_foundations/data/USvideos.csv
	new file:   part1_python_foundations/data/categories.csv
	new file:   part1_python_foundations/data/cleaned_video_records.csv
	new file:   part1_python_foundations/data/combined_videos.csv
	new file:   part1_python_foundations/notebooks/part1_python_foundations.ipynb
	new file:   part1_python_foundations/requirements.txt



In [101]:
%cd /content/TrendPulse
!git commit -m "Add Part 1 Python Foundations project"

/content/TrendPulse
[main 7488487] Add Part 1 Python Foundations project
 9 files changed, 138634 insertions(+)
 create mode 100644 part1_python_foundations/README.md
 create mode 100644 part1_python_foundations/data/GBvideos.csv
 create mode 100644 part1_python_foundations/data/INvideos.csv
 create mode 100644 part1_python_foundations/data/USvideos.csv
 create mode 100644 part1_python_foundations/data/categories.csv
 create mode 100644 part1_python_foundations/data/cleaned_video_records.csv
 create mode 100644 part1_python_foundations/data/combined_videos.csv
 create mode 100644 part1_python_foundations/notebooks/part1_python_foundations.ipynb
 create mode 100644 part1_python_foundations/requirements.txt


In [104]:
%cd /content/TrendPulse
!git push origin main

/content/TrendPulse
fatal: could not read Username for 'https://github.com': No such device or address


In [105]:
from google.colab import userdata
import subprocess
import base64

token = userdata.get("GITHUB_TOKEN")

auth = base64.b64encode(
    f"x-access-token:{token}".encode()
).decode()

result = subprocess.run(
    [
        "git", "-c",
        f"http.extraheader=AUTHORIZATION: basic {auth}",
        "push", "origin", "main"
    ],
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)


To https://github.com/karthikarajendiran2004-cyber/TrendPulse.git
   e8d92e5..7488487  main -> main



In [106]:
%cd /content/TrendPulse

with open("part1_python_foundations/README.md", "a") as file:
    file.write("""

## Validation Results
- Total unique records: 46,000
- Countries analysed: US, GB and IN
- Duplicate records removed: 230
- Missing numeric values handled.
- Date parsing validated successfully.
""")

print("README updated successfully!")

/content/TrendPulse
README updated successfully!


In [107]:
%cd /content/TrendPulse

!git add part1_python_foundations/README.md
!git commit -m "Update Part 1 project documentation"

/content/TrendPulse
[main 8c14590] Update Part 1 project documentation
 1 file changed, 8 insertions(+)


In [108]:
from google.colab import userdata
import subprocess
import base64

token = userdata.get("GITHUB_TOKEN")
auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()

result = subprocess.run(
    ["git", "-c", f"http.extraheader=AUTHORIZATION: basic {auth}",
     "push", "origin", "main"],
    capture_output=True, text=True
)

print(result.stdout)
print(result.stderr)


To https://github.com/karthikarajendiran2004-cyber/TrendPulse.git
   7488487..8c14590  main -> main

